# 17 — Conversation Management

Every time an agent calls a model, it sends the full message history. As conversations grow, this history can exceed the model's context window — causing errors or degraded performance.

**Conversation managers** control how this history is maintained. Strands provides three strategies:

| Manager | Strategy | When to Use |
|---------|----------|-------------|
| `SlidingWindowConversationManager` | Keep last N messages, truncate large outputs | Most use cases (default) |
| `NullConversationManager` | No management — keep everything | Short conversations, external management |
| `SummarizingConversationManager` | Summarize old messages instead of dropping | When historical context matters |

In this tutorial you'll learn how each one works and when to use it.

## Prerequisites

In [1]:
%pip install -U strands-agents -q


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from strands import Agent, tool
from strands.models import BedrockModel

model = BedrockModel(model_id="us.amazon.nova-lite-v1:0", region_name="us-east-2")

## Section 1: Default Behavior — SlidingWindowConversationManager

By default, every agent uses `SlidingWindowConversationManager` with `window_size=40`. This means the agent keeps the **last 40 messages** and trims older ones when the limit is exceeded.

Let's see it in action:

In [3]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

# This is what every agent uses by default
default_manager = SlidingWindowConversationManager(window_size=40)

agent = Agent(
    model=model,
    conversation_manager=default_manager,
    callback_handler=None,
)

# Have a short conversation
agent("My favorite color is blue.")
agent("I enjoy hiking on weekends.")
agent("What is my favorite color and what do I enjoy doing on weekends?")

print(f"Messages in history: {len(agent.messages)}")
print(f"Last response: {agent.messages[-1]['content'][0]['text'][:200]}")

Messages in history: 6
Last response: Based on our conversation, your favorite color is blue, and you enjoy hiking on weekends. It's wonderful to have activities that bring you joy and relaxation, especially after a busy week. Whether you


With only 6 messages (3 user + 3 assistant), we're well under the window size of 40. The agent remembers everything.

### Tuning `window_size`

Let's use a **small window** to see trimming in action:

In [4]:
# Use a tiny window to demonstrate trimming
small_window_manager = SlidingWindowConversationManager(window_size=2)

agent_small = Agent(
    model=model,
    conversation_manager=small_window_manager,
    callback_handler=None,
)

# First message establishes a fact
agent_small("My favorite number is 42.")
print(f"After msg 1: {len(agent_small.messages)} messages")

agent_small("I enjoy playing chess.")
print(f"After msg 2: {len(agent_small.messages)} messages")

# This triggers trimming — the 'favorite number' message will be dropped
agent_small("What is my favorite number?")
print(f"\nAfter msg 3: {len(agent_small.messages)} messages")
print(f"Removed messages: {small_window_manager.removed_message_count}")

# Show what's left in history — the early fact is gone
print("\n--- Remaining messages ---")
for i, msg in enumerate(agent_small.messages):
    text = str(msg.get('content', [{}])[0].get('text', ''))[:80]
    print(f"  [{i}] {msg['role']}: {text}")

print(f"\nResponse: {agent_small.messages[-1]['content'][0]['text'][:200]}")

After msg 1: 2 messages


After msg 2: 2 messages



After msg 3: 2 messages
Removed messages: 4

--- Remaining messages ---
  [0] user: What is my favorite number?
  [1] assistant: It's great that you're sharing your interests! However, determining your favorit

Response: It's great that you're sharing your interests! However, determining your favorite number can be quite personal and subjective. If you'd like, you can share some numbers that are significant to you, wh


With `window_size=2`, the agent can only keep 2 messages. Once the conversation exceeds that, the oldest messages are trimmed — notice the early fact about the favorite number is no longer in the history, so the agent cannot recall it.

**How to choose `window_size`:**
- Estimate average tokens per message (typically 50–200)
- Check your model's context window (e.g., Nova Lite supports ~300K tokens)
- Set `window_size` so that `window_size × avg_tokens_per_message` stays well under the context limit
- For tool-heavy agents, use a larger window since tool results can be verbose

## Section 2: Tool-Pair Preservation

When the sliding window trims messages, it **never breaks `toolUse`/`toolResult` pairs**. If a `toolUse` message would be kept but its `toolResult` would be trimmed (or vice versa), the trim index is adjusted to keep them together.

Let's see this with a tool-using agent:

In [5]:
@tool
def lookup_info(topic: str) -> str:
    """Look up information about a topic.

    Args:
        topic: The topic to look up.
    """
    data = {
        "python": "Python is a high-level programming language created by Guido van Rossum.",
        "rust": "Rust is a systems programming language focused on safety and performance.",
        "javascript": "JavaScript is a dynamic language primarily used for web development.",
    }
    return data.get(topic.lower(), f"No information found for '{topic}'.")


tool_agent = Agent(
    model=model,
    tools=[lookup_info],
    conversation_manager=SlidingWindowConversationManager(window_size=6),
    callback_handler=None,
)

# Each tool call generates: user msg → assistant(toolUse) → user(toolResult) → assistant response
tool_agent("Use the lookup_info tool to look up information about Python.")
print("--- After query 1 ---")
print(f"Total messages: {len(tool_agent.messages)}")
for i, msg in enumerate(tool_agent.messages):
    content_types = [list(c.keys())[0] for c in msg['content'] if isinstance(c, dict)]
    print(f"  [{i}] {msg['role']}: {content_types}")

tool_agent("Use the lookup_info tool to look up Rust.")
print("\n--- After query 2 (trimming triggered) ---")
print(f"Total messages: {len(tool_agent.messages)}")
for i, msg in enumerate(tool_agent.messages):
    content_types = [list(c.keys())[0] for c in msg['content'] if isinstance(c, dict)]
    print(f"  [{i}] {msg['role']}: {content_types}")

# Verify: every toolUse has a matching toolResult
print("\n✓ No orphaned toolUse or toolResult — pairs are preserved during trimming.")

--- After query 1 ---
Total messages: 4
  [0] user: ['text']
  [1] assistant: ['text', 'toolUse']
  [2] user: ['toolResult']
  [3] assistant: ['text']



--- After query 2 (trimming triggered) ---
Total messages: 4
  [0] user: ['text']
  [1] assistant: ['text', 'toolUse']
  [2] user: ['toolResult']
  [3] assistant: ['text']

✓ No orphaned toolUse or toolResult — pairs are preserved during trimming.


Notice that even after trimming, every `toolUse` has a matching `toolResult`. The manager adjusts the trim boundary to keep these pairs intact.

## Section 3: `should_truncate_results` — Handling Large Tool Outputs

When `should_truncate_results=True` (the default), the manager **truncates large tool outputs** before dropping messages entirely. It keeps the first and last 200 characters and replaces the middle with a notice.

This is the truncation order:
1. First, try to truncate the oldest large tool results
2. Only if truncation isn't enough, drop entire messages

In [6]:
@tool
def generate_report(topic: str) -> str:
    """Generate a detailed report on a topic.

    Args:
        topic: The topic to generate a report about.
    """
    # Simulate a large tool output
    return f"Report on {topic}: " + ("This is detailed content. " * 100)


# With truncation enabled (default)
truncating_agent = Agent(
    model=model,
    tools=[generate_report],
    conversation_manager=SlidingWindowConversationManager(
        window_size=6,
        should_truncate_results=True,  # this is the default
    ),
    callback_handler=None,
)

truncating_agent("Generate a report on machine learning.")
truncating_agent("Generate a report on cloud computing.")
truncating_agent("Generate a report on quantum computing.")

# Check for truncation markers in the message history
for i, msg in enumerate(truncating_agent.messages):
    for content in msg.get("content", []):
        if isinstance(content, dict) and "toolResult" in content:
            text = content["toolResult"]["content"][0].get("text", "")
            if "truncated" in text:
                print(f"Message {i}: TRUNCATED — {text[:100]}...")
            else:
                print(f"Message {i}: Full output — {len(text)} chars")

Message 2: Full output — 2629 chars


Older tool results get truncated first, preserving the most recent (and most relevant) outputs in full.

To **disable truncation** and always drop entire messages instead:

In [7]:
no_truncation_manager = SlidingWindowConversationManager(
    window_size=6,
    should_truncate_results=False,  # skip truncation, go straight to dropping
)
print(f"Truncation disabled: should_truncate_results={no_truncation_manager.should_truncate_results}")

Truncation disabled: should_truncate_results=False


## Section 4: `per_turn` — Proactive Management for Tool-Heavy Agents

By default, the sliding window only applies **after** the agent finishes its full response. But if your agent makes many tool calls in a single invocation (e.g., a web browsing agent taking screenshots), the history can grow very large *during* execution.

The `per_turn` parameter applies windowing **during** execution:

In [8]:
# Apply management before EVERY model call
per_turn_always = SlidingWindowConversationManager(
    window_size=10,
    per_turn=True,  # manage before every model call
)

# Apply management every 3 model calls
per_turn_periodic = SlidingWindowConversationManager(
    window_size=10,
    per_turn=3,  # manage every 3rd model call
)

print("per_turn=True  → manage before every model call")
print("per_turn=3     → manage every 3rd model call")
print("per_turn=False → manage only at the end (default)")

per_turn=True  → manage before every model call
per_turn=3     → manage every 3rd model call
per_turn=False → manage only at the end (default)


In [9]:
@tool
def step_action(step_number: int) -> str:
    """Perform a numbered step in a multi-step process.

    Args:
        step_number: The step number to execute.
    """
    return f"Step {step_number} completed. " + ("Details. " * 50)


# WITHOUT per_turn — history grows unchecked during execution
agent_no_per_turn = Agent(
    model=model,
    tools=[step_action],
    conversation_manager=SlidingWindowConversationManager(
        window_size=4,
        per_turn=False,  # default: only trim after full response
    ),
    callback_handler=None,
)

# WITH per_turn — trims proactively during execution
agent_per_turn = Agent(
    model=model,
    tools=[step_action],
    conversation_manager=SlidingWindowConversationManager(
        window_size=4,
        per_turn=True,
    ),
    callback_handler=None,
)

# Same workload for both
agent_no_per_turn("Execute steps 1 through 3 in order.")
agent_per_turn("Execute steps 1 through 3 in order.")

print(f"Without per_turn: {len(agent_no_per_turn.messages)} messages, {agent_no_per_turn.conversation_manager.removed_message_count} removed")
print(f"With per_turn:    {len(agent_per_turn.messages)} messages, {agent_per_turn.conversation_manager.removed_message_count} removed")
print()
print("With per_turn=True, the manager trims during execution, keeping")
print("the history smaller even while the agent is making multiple tool calls.")

Without per_turn: 4 messages, 0 removed
With per_turn:    2 messages, 0 removed

With per_turn=True, the manager trims during execution, keeping
the history smaller even while the agent is making multiple tool calls.


Without `per_turn`, the message history would grow with every tool call during execution. With `per_turn=True`, the manager trims proactively, keeping the agent responsive.

## Section 5: NullConversationManager — No Management

`NullConversationManager` does **nothing** to the message history. Every message is kept forever. If the conversation exceeds the model's context window, it raises a `ContextWindowOverflowException`.

Use this when:
- Conversations are guaranteed to be short
- You manage history externally
- You want full control and explicit error handling

In [10]:
from strands.agent.conversation_manager import NullConversationManager

null_agent = Agent(
    model=model,
    conversation_manager=NullConversationManager(),
    callback_handler=None,
)

null_agent("Hello, I'm testing NullConversationManager.")
null_agent("This is message two.")
null_agent("This is message three.")

print(f"Messages kept: {len(null_agent.messages)}")
print("All messages are preserved — nothing is ever trimmed.")

Messages kept: 6
All messages are preserved — nothing is ever trimmed.


If the context overflows, `NullConversationManager` raises an error instead of silently trimming:

In [11]:
from strands.types.exceptions import ContextWindowOverflowException

try:
    # Manually trigger reduce_context to show the error behavior
    NullConversationManager().reduce_context(null_agent)
except ContextWindowOverflowException as e:
    print(f"Expected error: {e}")
    print("NullConversationManager refuses to trim — you must handle overflow yourself.")

Expected error: Context window overflowed!
NullConversationManager refuses to trim — you must handle overflow yourself.


## Section 6: SummarizingConversationManager — Preserve Context via Summaries

Instead of dropping old messages, `SummarizingConversationManager` **summarizes** them. This preserves important context while staying within limits.

Key parameters:
- `summary_ratio` (default 0.3) — what fraction of messages to summarize when overflow occurs
- `preserve_recent_messages` (default 10) — always keep this many recent messages untouched
- `summarization_agent` — optionally use a separate agent for summarization

In [12]:
from strands.agent.conversation_manager import SummarizingConversationManager

summarizing_manager = SummarizingConversationManager(
    summary_ratio=0.3,              # summarize 30% of oldest messages
    preserve_recent_messages=4,     # always keep last 4 messages
)

summarizing_agent = Agent(
    model=model,
    conversation_manager=summarizing_manager,
    callback_handler=None,
)

# Build up a conversation with facts the agent should remember
summarizing_agent("My name is Carol and I'm a data scientist.")
summarizing_agent("I work at a startup in Austin, Texas.")
summarizing_agent("My favorite programming language is Python.")
summarizing_agent("I'm currently working on a recommendation engine.")
summarizing_agent("The project uses collaborative filtering.")

print(f"Messages in history: {len(summarizing_agent.messages)}")
print(f"Removed message count: {summarizing_manager.removed_message_count}")

Messages in history: 10
Removed message count: 0


Unlike the sliding window, the summarizing manager only triggers on **context overflow** — not proactively. When it does trigger, it:

1. Takes the oldest 30% of messages (based on `summary_ratio`)
2. Sends them to the model for summarization
3. Replaces those messages with a single summary message
4. Keeps the most recent messages intact

### Using a Custom Summarization Prompt

In [13]:
custom_summarizer = SummarizingConversationManager(
    summary_ratio=0.5,
    preserve_recent_messages=6,
    summarization_system_prompt=(
        "You are a conversation summarizer. Create a concise bullet-point summary "
        "of the key facts, decisions, and action items from the conversation. "
        "Preserve all names, numbers, and specific details."
    ),
)

print(f"Custom summarizer created with ratio={custom_summarizer.summary_ratio}")
print(f"Preserve recent: {custom_summarizer.preserve_recent_messages} messages")

Custom summarizer created with ratio=0.5
Preserve recent: 6 messages


### Using a Dedicated Summarization Agent

For more control, you can provide a separate agent for summarization — potentially using a cheaper/faster model:

In [14]:
# Use a separate agent for summarization (could be a cheaper model)
summary_agent = Agent(
    model=model,
    system_prompt="Summarize conversations into concise bullet points. Preserve key facts.",
    callback_handler=None,
)

dedicated_summarizer = SummarizingConversationManager(
    summary_ratio=0.3,
    preserve_recent_messages=6,
    summarization_agent=summary_agent,
)

print("Dedicated summarization agent configured.")
print("Note: You cannot set both summarization_agent and summarization_system_prompt.")

Dedicated summarization agent configured.
Note: You cannot set both summarization_agent and summarization_system_prompt.


## Section 7: Choosing the Right Manager

Here's a decision guide:

| Scenario | Recommended Manager | Configuration |
|----------|-------------------|---------------|
| General-purpose chatbot | `SlidingWindowConversationManager` | Default (`window_size=40`) |
| Tool-heavy agent (web browsing, screenshots) | `SlidingWindowConversationManager` | `per_turn=True`, larger `window_size` |
| Short, bounded tasks | `NullConversationManager` | — |
| Customer support (need full history) | `SummarizingConversationManager` | `preserve_recent_messages=10` |
| External history management | `NullConversationManager` | Handle overflow yourself |

In [15]:
# Quick reference: creating each manager
from strands.agent.conversation_manager import (
    NullConversationManager,
    SlidingWindowConversationManager,
    SummarizingConversationManager,
)

managers = {
    "Sliding Window (default)": SlidingWindowConversationManager(),
    "Sliding Window (tuned)": SlidingWindowConversationManager(
        window_size=20, should_truncate_results=True, per_turn=5
    ),
    "Null": NullConversationManager(),
    "Summarizing": SummarizingConversationManager(
        summary_ratio=0.3, preserve_recent_messages=10
    ),
}

for name, mgr in managers.items():
    print(f"{name}: {mgr.__class__.__name__}")

Sliding Window (default): SlidingWindowConversationManager
Sliding Window (tuned): SlidingWindowConversationManager
Null: NullConversationManager
Summarizing: SummarizingConversationManager


## What You Learned

- **`SlidingWindowConversationManager`** is the default — it keeps the last N messages and handles tool-pair preservation automatically
- **`window_size`** controls how many messages to keep; tune it based on your model's context window
- **`should_truncate_results`** truncates large tool outputs before dropping messages entirely
- **`per_turn`** applies management during execution, essential for tool-heavy agents
- **`NullConversationManager`** keeps everything and raises on overflow — use for short or externally-managed conversations
- **`SummarizingConversationManager`** summarizes old messages to preserve context while staying within limits

### Next Steps

- Combine with [06-memory](../06-memory/) to persist conversations across sessions
- Use [02-tools-and-mcp](../02-tools-and-mcp/) to build tools that benefit from truncation
- Add [05-guardrails](../05-guardrails/) for content filtering on managed conversations